# Project 2 — Veggie Classification with PyTorch

In this project you will classify images of vegetables into 10 categories using PyTorch.

You must build **two models**:
1. **A CNN from scratch** — design and train your own convolutional neural network
2. **A fine-tuned pretrained model** — take a model pretrained on ImageNet and adapt it for this task

## Deliverables

Submit the following to BrighSpace:
- Your github repository link
- Your saved model file (`.pth`) — must be under 400 MB
- A note (~1-2 paragraphs) explaining what you did to improve accuracy beyond a basic model

## Grading

| Component | Weight |
|---|---|
| Accuracy (validation set) | 60% |
| Code — readable and logical | 20% |
| Explanatory note | 20% |

In [ ]:
# Imports
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

from utils import get_device, show_images, plot_training_history, evaluate, per_class_accuracy

device = get_device()
print(f"Using device: {device}")

## Load the Data

Download `Vegetables.zip` from Moodle and place it in your working directory.

**On Colab**: You can drag-and-drop the zip into the file browser, or mount your Google Drive (see below).

The dataset contains:
- **Training**: 20,000 images (10 classes)
- **Validation**: 4,000 images (10 classes)

#### Temporary Files Warning
If you get errors loading images, remove temp files (e.g., `._` files on Mac):
- Mac: `dot_clean -n .` in the image folder
- Windows: search for and delete `._*` files

In [ ]:
# Optional: Load from Google Drive on Colab
# from google.colab import drive
# drive.mount('/content/drive')
# !cp "/content/drive/My Drive/Vegetables.zip" "Vegetables.zip"

In [ ]:
# Unzip the dataset
zip_name = "Vegetables.zip"

if not os.path.exists("Vegetables"):
    with zipfile.ZipFile(zip_name, "r") as zip_ref:
        zip_ref.extractall()
    print("Extracted Vegetables.zip")
else:
    print("Vegetables folder already exists")

In [ ]:
# Data directories
train_dir = "Vegetables/train"
val_dir = "Vegetables/validation"

IMAGE_SIZE = 224

# Define transforms — you may modify these to improve accuracy!
transform_train = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

transform_val = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

# Load datasets using ImageFolder (reads class names from subfolder names)
train_dataset = ImageFolder(train_dir, transform=transform_train)
val_dataset = ImageFolder(val_dir, transform=transform_val)

class_names = train_dataset.classes

print(f"Training samples:   {len(train_dataset):,}")
print(f"Validation samples: {len(val_dataset):,}")
print(f"Classes ({len(class_names)}): {class_names}")

In [ ]:
# Create data loaders
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
# Visualize some training images
viz_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])
viz_dataset = ImageFolder(train_dir, transform=viz_transform)
indices = torch.randperm(len(viz_dataset))[:8]
viz_images = torch.stack([viz_dataset[i][0] for i in indices])
viz_labels = [viz_dataset[i][1] for i in indices]
show_images(viz_images, viz_labels, class_names)

---

## Part 1: Build a CNN from Scratch

Design your own CNN architecture. Some things to consider:
- How many convolutional layers/blocks do you need?
- What filter sizes and counts will you use?
- Will you use batch normalization? Dropout?
- What pooling strategy — max pooling, global average pooling, or both?

**Hints**:
- Start simple (2-3 conv blocks) and add complexity if needed
- Batch normalization helps training stability
- Dropout helps prevent overfitting
- Global average pooling before the classifier reduces parameter count

In [ ]:
# TODO: Define your CNN model
class VeggieCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # YOUR CODE HERE
        # Build your convolutional feature extractor and classifier
        pass

    def forward(self, x):
        # YOUR CODE HERE
        pass


scratch_model = VeggieCNN().to(device)
print(scratch_model)

# Print parameter count
total_params = sum(p.numel() for p in scratch_model.parameters())
print(f"\nTotal parameters: {total_params:,}")

In [ ]:
# TODO: Set up your training — choose optimizer, loss function, and hyperparameters
NUM_EPOCHS = 10  # adjust as needed

criterion = nn.CrossEntropyLoss()
# optimizer = ...
# scheduler = ... (optional but recommended)

In [ ]:
# TODO: Write your training loop
# Track train_losses, val_losses, train_accs, val_accs for plotting

train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(1, NUM_EPOCHS + 1):
    # --- Training phase ---
    # YOUR CODE HERE

    # --- Validation phase ---
    # Hint: use evaluate() from utils.py

    pass

In [ ]:
# Plot your training curves
# plot_training_history(train_losses, val_losses, train_accs, val_accs)

In [ ]:
# Show per-class accuracy
# per_class_accuracy(scratch_model, val_loader, class_names, device)

---

## Part 2: Fine-Tune a Pretrained Model

Load a pretrained model (e.g., ResNet-18) and adapt it for veggie classification.

Steps:
1. Load the pretrained model with `models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)`
2. Replace the final classification layer for 10 classes
3. Choose a training strategy:
   - **Feature extraction**: Freeze all pretrained layers, train only the new head
   - **Fine-tuning**: Train the entire network with a lower learning rate for pretrained layers

**Hints**:
- Use a lower learning rate for pretrained layers than for the new head
- Data augmentation (random flips, rotations, color jitter) can help a lot
- A learning rate scheduler can improve final accuracy
- You can try other models: `resnet50`, `mobilenet_v3_small`, `efficientnet_b0`

In [ ]:
# TODO: Load and modify a pretrained model
# pretrained_model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Replace the final layer:
# pretrained_model.fc = nn.Linear(pretrained_model.fc.in_features, 10)

# pretrained_model = pretrained_model.to(device)

In [ ]:
# TODO: Set up optimizer (consider differential learning rates)
# TODO: Train the model
# TODO: Plot results

---

## Compare Your Models

Fill in the table below with your results.

| | CNN from Scratch | Pretrained (Fine-tuned) |
|---|---|---|
| Parameters | ??? | ??? |
| Epochs trained | ??? | ??? |
| Best val accuracy | ???% | ???% |
| Training time | ??? | ??? |

---

## Save Your Best Model

Save your best model for submission. The file must be **under 400 MB** for Moodle.

In [ ]:
# Save your best model (pick whichever performed better)
# best_model = ...  # scratch_model or pretrained_model

# torch.save(best_model.state_dict(), "veggie_model.pth")

# Check file size
# import os
# size_mb = os.path.getsize("veggie_model.pth") / (1024 * 1024)
# print(f"Model size: {size_mb:.1f} MB")
# assert size_mb < 400, f"Model too large for Moodle! ({size_mb:.1f} MB > 400 MB)"

---

## Tips for Improving Accuracy

Here are some strategies to try (mention what you used in your explanatory note!):

**Data augmentation** — add to your training transforms:
- `transforms.RandomHorizontalFlip()`
- `transforms.RandomRotation(15)`
- `transforms.ColorJitter(brightness=0.2, contrast=0.2)`
- `transforms.RandomResizedCrop(224, scale=(0.8, 1.0))`

**Architecture choices**:
- Add batch normalization after each conv layer
- Use dropout (0.3-0.5) to reduce overfitting
- Try deeper or wider networks

**Training tricks**:
- Learning rate schedulers (`CosineAnnealingLR`, `ReduceLROnPlateau`)
- Weight decay in the optimizer (e.g., `weight_decay=1e-4`)
- Train for more epochs if not overfitting
- Use `AdamW` instead of `Adam`